# KNOCK — Long Music Video Generation Pipeline (Vertex AI)

Refactored to use Google Cloud Vertex AI (Imagen 3 & Veo) instead of local Diffusers.

In [19]:
#@title 1. Install Dependencies
!pip install -q -U google-genai imageio imageio-ffmpeg moviepy opencv-python pillow pyyaml tqdm google-cloud-storage


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
#@title 2. Global Config
import os
import json
import math
import random
import time
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# --- AI STUDIO CONFIGURATION ---
# Make sure your GEMINI_API_KEY GCP_PROJECT_ID GCP_LOCATION environment variable is set in .env file!
GCP_PROJECT_ID = os.environ["GCP_PROJECT_ID"]
GCP_LOCATION = os.environ["GCP_LOCATION"]

# --- PIPELINE SWITCHES ---
GENERATE_CHARACTER_REFERENCES = False
CREATE_SCRIPTS = True
GENERATE_KEYFRAMES = True
GENERATE_VIDEO_CLIPS = True
BUILD_SCENES = True
BUILD_FINAL_VIDEO = True
USE_CHARACTER_REFERENCES = True
MAX_SCENES_TO_RUN = None

# --- VIDEO SETTINGS ---
OUTPUT_FPS = 24
FINAL_WIDTH = 1024
FINAL_HEIGHT = 576
CROSSFADE_SECONDS = 1.2
MAX_SLOWDOWN_FACTOR = 2.0
FADE_TO_BLACK_SECONDS = 2.0
ADD_FILM_GRAIN = True
ADD_SUBTITLES = True
ADD_LETTERBOX = True
ADD_SLOW_ZOOM = True

AUDIO_PATH = "Bob_Dylan_-_Knockin_On_Heaven_s_Door.mp3"
GLOBAL_SEED = 358
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


#Vertex api connection test


In [2]:
#@title 3. Setup Folders
from pathlib import Path

PROJECT_DIR = Path('/home/bne/Projects/Knock')
KEYFRAME_DIR = PROJECT_DIR / 'keyframes'
CLIP_DIR = PROJECT_DIR / 'veo_clips'
SCENE_DIR = PROJECT_DIR / 'scenes'
FINAL_DIR = PROJECT_DIR / 'final'
ASSETS_DIR = PROJECT_DIR / 'assets'
CHARACTER_DIR = ASSETS_DIR / 'character_refs'
SCRIPT_DIR = ASSETS_DIR / 'scripts'

for d in [KEYFRAME_DIR, CLIP_DIR, SCENE_DIR, FINAL_DIR, ASSETS_DIR, CHARACTER_DIR, SCRIPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directories created in:', PROJECT_DIR)

Directories created in: /home/bne/Projects/Knock


In [ ]:
#@title 3.5. AI Script Creation (Gemini 2.5 Flash)
if CREATE_SCRIPTS:
    from google import genai
    from google.genai import types

    SCENARIO_PATH = PROJECT_DIR / 'scenario.md'
    CHARACTERS_PATH = SCRIPT_DIR / 'CHARACTERS.json'
    SCENES_PATH = SCRIPT_DIR / 'SCENES.json'

    scenario_text = SCENARIO_PATH.read_text(encoding='utf-8').strip()
    if not scenario_text:
        print("⚠️  scenario.md is empty. Skipping script creation.")
    else:
        client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

        system_prompt = """You are an expert cinematographer and AI prompt engineer specializing in diffusion model image generation (Imagen 3) and video generation (Veo 2).

Your job is to read a narrative scenario — which may be a plain paragraph, rough notes, a story outline, a detailed brief or very close to structured json we want — and produce two structured JSON files that will drive an AI music video generation pipeline.

You must output:
1. CHARACTERS.json — defining all characters visually for diffusion models
2. SCENES.json — breaking the narrative into individual cinematic scenes with diffusion-optimized prompts

ALWAYS UPHOLD THIS RULES:
- Extract character identities, visual styles, and emotional tones from the scenario text
- Break the narrative into logical cinematic scenes that fit the emotional arc
- Write prompts in concrete, visual, diffusion-model-friendly language (avoid abstract metaphors)
- Write comprehensive negative prompts to block common diffusion artifacts
- Write detailed motion hints for Veo 2 video generation (camera moves, character actions, atmosphere)
- Keep emotional and narrative intent faithful to the scenario
- scene duration fields should be reasonable integers in seconds
- character_reference in scenes must match one of the ids defined in CHARACTERS character_references, or be null
-if scenario is close or same structured as scene and character json enhence prompting for diffusion models
- All path fields in character_references must follow this pattern: CHARACTER_DIR_PLACEHOLDER/<filename>.png
The output MUST follow this exact structure:

===CHARACTERS_JSON===
[
  {
    "name": "string — character full name or title",
    "core_description": "string — concise visual identity sentence optimized for diffusion models",
    "visual_markers": ["concrete visual descriptor 1", "visual descriptor 2", "..."],
    "negative_markers": ["thing to exclude 1", "thing to exclude 2", "..."],
    "character_references": [
      {
        "id": "string — short snake_case identifier e.g. adult_front",
        "path": "string — absolute file path",
        "prompt": "string — detailed diffusion prompt to generate this specific reference image angle/pose"
      }
    ]
  }
]
===SCENES_JSON===
[
  {
    "id": 1,
    "title": "string — short evocative scene title",
    "duration": 6,
    "start_time": "0:00",
    "end_time": "0:06",
    "lyric": "string — lyric line or 'Instrumental' for this moment",
    "emotion": "string — one or two word emotional quality",
    "character_reference": "string id matching a character_references id, or null",
    "prompt": "string — detailed diffusion model prompt for the keyframe image",
    "negative_prompt": "string — comprehensive comma-separated negative prompt",
    "motion_hint": "string — detailed Veo 2 camera motion, character actions, atmosphere"
  }
]

Return ONLY the markers and valid JSON. No explanation text.""".replace("CHARACTER_DIR_PLACEHOLDER", str(CHARACTER_DIR))

        user_message = f"""Here is the scenario for the music video:

---SCENARIO---
{scenario_text}
---END SCENARIO---

Read this scenario carefully and create the full CHARACTERS.json and SCENES.json for the AI video generation pipeline."""

        print("📝 Sending scenario to Gemini 2.5 Flash for script creation...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[user_message],
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.7,
            )
        )
        print("response came from gemini starting to parse")
        result_text = response.text.strip()

        try:
            chars_part = result_text.split('===CHARACTERS_JSON===')[1].split('===SCENES_JSON===')[0].strip()
            scenes_part = result_text.split('===SCENES_JSON===')[1].strip()

            # Strip markdown code fences if present
            for fence in ['```json', '```']:
                chars_part = chars_part.replace(fence, '').strip()
                scenes_part = scenes_part.replace(fence, '').strip()

            created_characters = json.loads(chars_part)
            created_scenes = json.loads(scenes_part)

            with open(CHARACTERS_PATH, 'w') as f:
                json.dump(created_characters, f, indent=4)
            with open(SCENES_PATH, 'w') as f:
                json.dump(created_scenes, f, indent=4)

            print(f"✅ Created {len(created_characters)} character(s) and {len(created_scenes)} scene(s) from scenario.")
            print(f"   Saved to CHARACTERS.json and SCENES.json")
        except (IndexError, json.JSONDecodeError) as e:
            print(f"❌ Failed to parse Gemini response: {e}")
            print("Raw response preview:")
            print(result_text[:800])


📝 Sending scenario to Gemini 2.5 Flash for script creation...
✅ Created 1 character(s) and 20 scene(s) from scenario.
   Saved to CHARACTERS.json and SCENES.json


In [16]:
#@title 4. Load Scene Data & Characters
with open(SCRIPT_DIR / 'CHARACTERS.json', 'r') as f:
    CHARACTERS = json.load(f)

with open(SCRIPT_DIR / 'SCENES.json', 'r') as f:
    SCENES = json.load(f)

# Build lookup structures for downstream cells
CHARACTER_BIBLE = CHARACTERS[0]
CHARACTER_REFERENCES = {
    ref["id"]: ref["path"] for ref in CHARACTER_BIBLE["character_references"]
}
CHARACTER_REFERENCE_PROMPTS = {
    ref["id"]: ref["prompt"] for ref in CHARACTER_BIBLE["character_references"]
}

if MAX_SCENES_TO_RUN is not None:
    SCENES = SCENES[:MAX_SCENES_TO_RUN]

print(f"Loaded {len(CHARACTERS)} character(s), {len(SCENES)} scene(s).")
print(f"Character references: {list(CHARACTER_REFERENCES.keys())}")


Loaded 1 character(s), 20 scene(s).
Character references: ['adult_front', 'adult_back', 'adult_silhouette', 'adult_side', 'child_academy']


In [10]:
#@title 5. Utility Functions
def resize_and_crop_pil(img, width=FINAL_WIDTH, height=FINAL_HEIGHT):
    img = img.convert('RGB')
    src_w, src_h = img.size
    scale = max(width / src_w, height / src_h)
    new_w, new_h = int(src_w * scale), int(src_h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - width) // 2
    top = (new_h - height) // 2
    return img.crop((left, top, left + width, top + height))

def pil_to_cv2(img):
    return cv2.cvtColor(np.array(img.convert('RGB')), cv2.COLOR_RGB2BGR)

def cv2_to_pil(frame):
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

def add_film_grain_frame(frame, strength=8):
    if not ADD_FILM_GRAIN: return frame
    noise = np.random.normal(0, strength, frame.shape).astype(np.int16)
    return np.clip(frame.astype(np.int16) + noise, 0, 255).astype(np.uint8)

def add_letterbox_frame(frame, bar_ratio=0.08):
    if not ADD_LETTERBOX: return frame
    h, w = frame.shape[:2]
    bar = int(h * bar_ratio)
    out = frame.copy()
    out[:bar, :] = 0
    out[h-bar:, :] = 0
    return out

def draw_subtitle(frame, text, scene_title=None):
    if not ADD_SUBTITLES or not text: return frame
    img = cv2_to_pil(frame)
    draw = ImageDraw.Draw(img)
    w, h = img.size
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf', 32)
        small_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 18)
    except:
        font = ImageFont.load_default()
        small_font = ImageFont.load_default()

    bbox = draw.textbbox((0, 0), text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    x = (w - text_w) // 2
    y = h - 78
    pad = 14
    overlay = Image.new('RGBA', img.size, (0,0,0,0))
    odraw = ImageDraw.Draw(overlay)
    odraw.rounded_rectangle((x-pad, y-pad, x+text_w+pad, y+text_h+pad), radius=10, fill=(0,0,0,110))
    img = Image.alpha_composite(img.convert('RGBA'), overlay)
    draw = ImageDraw.Draw(img)
    draw.text((x, y), text, font=font, fill=(245, 238, 220, 255))
    if scene_title:
        draw.text((34, 28), scene_title.upper(), font=small_font, fill=(210, 200, 180, 180))
    return pil_to_cv2(img.convert('RGB'))

def slow_zoom_frame(frame, t, total_frames, max_zoom=1.08):
    if not ADD_SLOW_ZOOM: return frame
    h, w = frame.shape[:2]
    zoom = 1.0 + (max_zoom - 1.0) * (t / max(1, total_frames - 1))
    new_w, new_h = int(w / zoom), int(h / zoom)
    x1 = (w - new_w) // 2
    y1 = (h - new_h) // 2
    cropped = frame[y1:y1+new_h, x1:x1+new_w]
    return cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)

def read_video_frames(video_path):
    cap = cv2.VideoCapture(str(video_path))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.resize(frame, (FINAL_WIDTH, FINAL_HEIGHT))
        frames.append(frame)
    cap.release()
    return frames

def write_video_frames(frames, output_path, fps=24):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (FINAL_WIDTH, FINAL_HEIGHT))
    for f in frames: writer.write(f)
    writer.release()

def build_prompt(scene):
    markers = ", ".join(CHARACTER_BIBLE["visual_markers"])
    neg_markers = ", ".join(CHARACTER_BIBLE["negative_markers"])
    pos = f"{scene['prompt']}. Main character: {CHARACTER_BIBLE['core_description']}. Visual markers: {markers}. Cinematic 1970s film still."
    neg = f"{neg_markers}, {scene.get('negative_prompt', '')}, extra people, text, watermark"
    return pos, neg

def get_reference_path_for_scene(scene):
    ref_key = scene.get("character_reference")
    return CHARACTER_REFERENCES.get(ref_key) if ref_key else None

In [ ]:
#@title 6. Stage 1: Keyframe Generation (Gemini Image via AI Studio)
if GENERATE_KEYFRAMES:
    import time
    from google import genai
    from google.genai import types
    from PIL import Image
    from pathlib import Path
    
    print("Loading Gemini Image endpoint (AI Studio)...")
    #  use this client if want google ai studio and GEMINI_API_KEY from environment
    #  client = genai.Client()
    client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

    if GENERATE_CHARACTER_REFERENCES and USE_CHARACTER_REFERENCES:
        missing_refs = [k for k, p in CHARACTER_REFERENCES.items() if not Path(p).exists()]
        for idx, ref_key in enumerate(missing_refs):
            out_path = CHARACTER_REFERENCES[ref_key]
            prompt = CHARACTER_REFERENCE_PROMPTS[ref_key] + " Create this in a 16:9 aspect ratio."
            print(f"Generating reference: {ref_key}")
            try:
                response = client.models.generate_content(
                    model='gemini-3.1-flash-image-preview',
                    contents=[prompt],
                )
                for part in response.parts:
                    if part.inline_data is not None:
                        img = part.as_image()
                        img.save(out_path)
                        # Resize and crop to ensure exact dimensions
                        img = Image.open(out_path)
                        resize_and_crop_pil(img).save(out_path)
                        print(f"Saved: {out_path}")
                        break
            except Exception as e:
                print(f"API Error: {e}")
            time.sleep(2)

    for scene in tqdm(SCENES, desc="Generating Scene Keyframes"):
        out_path = KEYFRAME_DIR / f"scene_{scene['id']:02d}.png"
        if out_path.exists(): continue

        prompt, negative_prompt = build_prompt(scene)
        full_prompt = f"{prompt}\n\nNegative aspects to avoid: {negative_prompt}\n\nGenerate this in a 16:9 aspect ratio."

        try:
            contents = [full_prompt]
            ref_path = get_reference_path_for_scene(scene)
            
            if USE_CHARACTER_REFERENCES and ref_path and Path(ref_path).exists():
                print(f"Uploading reference image to AI Studio ==> {ref_path}")
                # AI Studio uses the built-in File API instead of GCS!
                ref_file = client.files.upload(file=str(ref_path))
                contents.insert(0, ref_file)
                
            response = client.models.generate_content(
                model='gemini-3.1-flash-image-preview',
                contents=contents,
            )
            
            for part in response.parts:
                if part.inline_data is not None:
                    img = part.as_image()
                    img.save(out_path)
                    
                    img = Image.open(out_path)
                    resize_and_crop_pil(img).save(out_path)
                    print(f"Saved: {out_path}")
                    break
        except Exception as e:
            print(f"API Error on scene {scene['id']}: {e}")
            
        time.sleep(3)


Loading Imagen 3 endpoint...


/home/bne/Projects/Knock/venv/lib/python3.14/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
Generating Scene Keyframes:   0%|          | 0/9 [00:00<?, ?it/s]/home/bne/Projects/Knock/venv/lib/python3.14/site-packages/vertexai/vision_models/_vision_models.py:154: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


refernce path==> 
/home/bne/Projects/Knock/assets/character_refs/former_player_side.png


/home/bne/Projects/Knock/venv/lib/python3.14/site-packages/vertexai/vision_models/_vision_models.py:1437: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
/home/bne/Projects/Knock/venv/lib/python3.14/site-packages/vertexai/vision_models/_vision_models.py:154: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()
DEBUG:PIL.PngImagePlugin:STREAM b'IHDR' 16 13
DEBUG:PIL.PngImagePlugin:STREAM b'zTXt' 41 137
DEBUG:PIL.PngImagePlugin:STREAM b'iTXt' 190 647
DEBUG:PIL.PngImagePlugin:STREAM b'caBX' 849 6167
DEBUG:PIL.PngImagePlugin:b'caBX' 849 6167 (unknown)
DEBUG:PIL.PngImagePlugin:STREAM b'IDAT' 7028 8192


Saved: /home/bne/Projects/Knock/keyframes/scene_08.png


Generating Scene Keyframes:  89%|████████▉ | 8/9 [00:18<00:02,  2.29s/it]

API Error on scene 9: 400 Image generation failed with the following error: Model 'imagen-3.0-capability-001' is invalid endpoint


Generating Scene Keyframes: 100%|██████████| 9/9 [00:21<00:00,  2.43s/it]


In [11]:
#@title 7. Stage 2: Video Generation (Veo via Vertex AI)
if GENERATE_VIDEO_CLIPS:
    import time
    from tqdm.auto import tqdm
    from google import genai
    from google.genai import types
    from google.cloud import storage

    print("Loading Veo endpoint (Vertex AI)...")
    client = genai.Client(vertexai=True, project=GCP_PROJECT_ID, location=GCP_LOCATION)

    # Set up GCS bucket for Veo output
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = f"{GCP_PROJECT_ID}-veo-output"
    bucket = storage_client.bucket(bucket_name)
    if not bucket.exists():
        try:
            bucket.create(location=GCP_LOCATION)
            print(f"Created GCS bucket: {bucket_name}")
        except Exception as e:
            print(f"Could not create GCS bucket: {e}")
    gcs_output_uri = f"gs://{bucket_name}/"

    for scene in tqdm(SCENES, desc="Generating Video Clips via Veo"):
        keyframe_path = KEYFRAME_DIR / f"scene_{scene['id']:02d}.png"
        final_clip_path = CLIP_DIR / f"scene_{scene['id']:02d}_veo.mp4"

        if final_clip_path.exists():
            print(f'Skipping existing Veo clip: {final_clip_path}')
            continue
        if not keyframe_path.exists():
            print(f'Missing keyframe for scene {scene["id"]}. Skipping.')
            continue

        try:
            with open(str(keyframe_path), "rb") as f:
                image_bytes = f.read()

            source_image = types.Image(
                image_bytes=image_bytes,
                mime_type="image/png"
            )

            motion_prompt = f"Cinematic motion: {scene.get('motion_hint', 'slow, steady camera movement')}."

            print(f"Requesting Veo generation for scene {scene['id']}...")
            duration_seconds=scene.get('duration', 5)  
            if(duration_seconds<5):
                duration_seconds=5
            # Add this right before the generate_videos call to see exactly what Veo receives
            print(f"Scene {scene['id']} prompt sent to Veo: {motion_prompt}")

            operation = client.models.generate_videos(
                model='veo-2.0-generate-001',
                prompt=motion_prompt,
                image=source_image,
                config=types.GenerateVideosConfig(
                    aspect_ratio="16:9",
                    fps=OUTPUT_FPS,
                    duration_seconds=duration_seconds,
                    output_gcs_uri=gcs_output_uri,
                    person_generation="allow_all"
                )
            )

            while not operation.done:
                time.sleep(15)
                print(f"  Polling scene {scene['id']}...")
                operation = client.operations.get(operation=operation)

            if operation.error:
                print(f"API Error on video scene {scene['id']}: {operation.error}")
                continue

            if operation.response and operation.response.generated_videos:
                video = operation.response.generated_videos[0]
                # Download from GCS
                uri = video.video.uri
                b_name = uri.split('/')[2]
                blob_name = '/'.join(uri.split('/')[3:])
                blob = storage_client.bucket(b_name).blob(blob_name)
                blob.download_to_filename(str(final_clip_path))
                print(f'Downloaded Veo clip: {final_clip_path}')
        except Exception as e:
            print(f"API Error on video scene {scene['id']}: {e}")

        time.sleep(10)


Loading Veo endpoint (Vertex AI)...


Generating Video Clips via Veo:   0%|          | 0/20 [00:00<?, ?it/s]

Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_01_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_02_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_03_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_04_veo.mp4
Requesting Veo generation for scene 5...
Scene 5 prompt sent to Veo: Cinematic motion: Static wide shot,  floodlight turns off then slow fade of the environment into deep shadow..
  Polling scene 5...
  Polling scene 5...
Downloaded Veo clip: /home/bne/Projects/Knock/veo_clips/scene_05_veo.mp4


Generating Video Clips via Veo:  25%|██▌       | 5/20 [00:49<02:27,  9.82s/it]

Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_06_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_07_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_08_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_09_veo.mp4
Requesting Veo generation for scene 10...
Scene 10 prompt sent to Veo: Cinematic motion: wakes up slowly, reaching motion toward the floor and falling down, slight handheld camera shake reflecting his internal state..
  Polling scene 10...
  Polling scene 10...
Downloaded Veo clip: /home/bne/Projects/Knock/veo_clips/scene_10_veo.mp4


Generating Video Clips via Veo: 100%|██████████| 20/20 [01:38<00:00,  4.94s/it]

Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_11_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_12_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_13_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_14_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_15_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_16_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_17_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_18_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_19_veo.mp4
Skipping existing Veo clip: /home/bne/Projects/Knock/veo_clips/scene_20_veo.mp4


In [17]:
#@title 8. Stage 3 & 4: Trim & Composition
def parse_time(t_str):
    """Parse 'M:SS' to seconds."""
    parts = t_str.split(':')
    return int(parts[0]) * 60 + int(parts[1])

def get_scene_duration(scene):
    """Calculate actual duration from start_time and end_time."""
    return parse_time(scene['end_time']) - parse_time(scene['start_time'])

def trim_frames(src_frames, target_seconds, fps=OUTPUT_FPS):
    """Trim frames to exact target duration. If clip is shorter, keep all frames."""
    target_count = int(target_seconds * fps)
    if len(src_frames) <= target_count:
        return src_frames
    return src_frames[:target_count]

def build_scene(scene, input_clip_path, output_scene_path):
    src_frames = read_video_frames(input_clip_path)
    if not src_frames: return
    
    actual_duration = get_scene_duration(scene)
    target_count = int(actual_duration * OUTPUT_FPS)
    
    # Trim the Veo clip to the exact needed duration
    trimmed = trim_frames(src_frames, actual_duration)
    
    # If clip is still shorter than needed, resample to fill
    if len(trimmed) < target_count:
        trimmed = resample_without_rewind(trimmed, target_count)
    
    # Decorate each frame
    decorated = [decorate_scene_frame(f, scene, i, target_count) for i, f in enumerate(trimmed)]
    
    write_video_frames(decorated[:target_count], output_scene_path, fps=OUTPUT_FPS)
    print(f"  Scene {scene['id']}: {len(src_frames)} source frames -> trimmed to {target_count} frames ({actual_duration}s)")

def resample_without_rewind(src_frames, output_count):
    if output_count <= 0: return []
    if len(src_frames) == 1: return [src_frames[0].copy() for _ in range(output_count)]
    out = []
    n = len(src_frames)
    for i in range(output_count):
        pos = i * (n - 1) / max(1, output_count - 1)
        lo = int(math.floor(pos))
        hi = min(n - 1, lo + 1)
        t = pos - lo
        a = src_frames[lo].astype(np.float32)
        b = src_frames[hi].astype(np.float32)
        out.append(np.clip((1 - t) * a + t * b, 0, 255).astype(np.uint8))
    return out

def decorate_scene_frame(frame, scene, i, target_count, subtitle_allowed=True):
    frame = slow_zoom_frame(frame, i, target_count)
    frame = add_film_grain_frame(frame)
    frame = add_letterbox_frame(frame)
    lyric = scene['lyric'] if (subtitle_allowed and i > OUTPUT_FPS * 2 and i < target_count - OUTPUT_FPS * 2) else ""
    return draw_subtitle(frame, lyric, scene['title'])

if BUILD_SCENES:
    from tqdm.auto import tqdm
    for scene in tqdm(SCENES, desc="Trimming & Processing Scenes"):
        clip_path = CLIP_DIR / f"scene_{scene['id']:02d}_veo.mp4"
        out_path = SCENE_DIR / f"scene_{scene['id']:02d}_extended.mp4"
        if out_path.exists(): continue
        if clip_path.exists(): build_scene(scene, clip_path, out_path)
if BUILD_FINAL_VIDEO:
    def fade_to_black_and_in(a_frames, b_frames, fade_count):
        if fade_count <= 0 or not a_frames or not b_frames: return a_frames + b_frames
        fade_out_count = min(fade_count, len(a_frames))
        fade_in_count = min(fade_count, len(b_frames))
        
        res = a_frames[:-fade_out_count]
        
        # Fade out a_frames to black
        for i in range(fade_out_count):
            alpha = 1.0 - ((i + 1) / fade_out_count)
            a = a_frames[-fade_out_count + i].astype(np.float32)
            res.append(np.clip(a * alpha, 0, 255).astype(np.uint8))
            
        # Fade in b_frames from black
        for i in range(fade_in_count):
            alpha = (i + 1) / fade_in_count
            b = b_frames[i].astype(np.float32)
            res.append(np.clip(b * alpha, 0, 255).astype(np.uint8))
            
        res.extend(b_frames[fade_in_count:])
        return res

    scene_paths = [SCENE_DIR / f"scene_{s['id']:02d}_extended.mp4" for s in SCENES if (SCENE_DIR / f"scene_{s['id']:02d}_extended.mp4").exists()]
    if scene_paths:
        final_frames = []
        fade_frames = int(CROSSFADE_SECONDS * OUTPUT_FPS)
        for i, path in enumerate(scene_paths):
            frames = read_video_frames(path)
            final_frames = frames if i == 0 else fade_to_black_and_in(final_frames, frames, fade_frames)
        
        no_audio_path = FINAL_DIR / 'knock_ai_video_no_audio.mp4'
        write_video_frames(final_frames, no_audio_path, OUTPUT_FPS)
        
        try:
            from moviepy import VideoFileClip, AudioFileClip
            if AUDIO_PATH and Path(AUDIO_PATH).exists():
                video = VideoFileClip(str(no_audio_path))
                audio_clip = AudioFileClip(str(AUDIO_PATH))
                
                # Start audio at 17s, ending at 17s + video duration
                end_time = min(audio_clip.duration, 17 + video.duration)
                audio = audio_clip.subclipped(17, end_time)
                
                final = video.with_audio(audio)
                final.write_videofile(str(FINAL_DIR / 'knock_ai_video_with_audio.mp4'), codec='libx264', audio_codec='aac', fps=OUTPUT_FPS)
                video.close(); audio.close(); final.close()
                print("Finished building final video with audio.")
            else:
                print("Finished building silent video (audio file not found).")
        except ImportError:
            print("Finished building silent video. MoviePy not installed for audio injection.")


Trimming & Processing Scenes: 100%|██████████| 20/20 [00:07<00:00,  2.55it/s]

  Scene 20: 192 source frames -> trimmed to 192 frames (8s)


MoviePy - Building video /home/bne/Projects/Knock/final/knock_ai_video_with_audio.mp4.
MoviePy - Writing audio in knock_ai_video_with_audioTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video /home/bne/Projects/Knock/final/knock_ai_video_with_audio.mp4



MoviePy - Done !
MoviePy - video ready /home/bne/Projects/Knock/final/knock_ai_video_with_audio.mp4
Finished building final video with audio.
